# Improve weighted XGBoost using validation data

Baseline: 25 features, positive weight 7, threshold 0.50, learning rate 0.03, depth 3, 400 trees, and row/column sampling 0.8.

Run cells in order with the project's `mlops` Python environment. This staged search trains 21 models by default. It compares feature counts, searches 12 depth/child-weight combinations, tries two learning rates with early stopping, and compares five class weights. Model candidates are ranked by validation average precision; the final threshold is chosen by validation F1. This is a staged search, not an exhaustive joint optimization.

The test split is not loaded during tuning. Repeated validation searches can overfit validation data; a final test comparison is optional and must not be used to pick another configuration.

MLflow stores every candidate's parameters, training/validation metrics and fitted pipeline in experiment `churn-xgboost-weighted-improvement`, using `sqlite:///mlflow_churn.db`. A shared `search_id` tag groups each notebook execution. A separate selected-model run stores the chosen threshold, comparison tables and model; optional test results attach to that run. Each run closes automatically, including on failure.

From the project directory, open the tracking UI with `mlflow ui --backend-store-uri sqlite:///mlflow_churn.db`.


In [1]:
from pathlib import Path
import json
import os
import pickle
from uuid import uuid4
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, confusion_matrix,
)
from xgboost import XGBClassifier

DATA_PATH = Path("data/processed_datasets.npz")
OUTPUT_DIR = Path("data/xgboost_weighted_improvement")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with np.load(DATA_PATH) as data:
    X_train, y_train = data["X_train"], data["Y_train"]
    X_val, y_val = data["X_val"], data["Y_val"]
assert X_train.shape[1] == X_val.shape[1]
assert set(np.unique(y_train)) == {0, 1}
assert set(np.unique(y_val)) == {0, 1}
print("Train:", X_train.shape, "Validation:", X_val.shape)

BASE_PARAMS = dict(
    learning_rate=0.03, max_depth=3, n_estimators=400,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=7,
    min_child_weight=1, random_state=42,
    n_jobs=min(8, os.cpu_count() or 1), eval_metric="aucpr",
)
FEATURE_COUNTS = list(dict.fromkeys([25, X_train.shape[1]]))
DEPTHS = [3, 4, 5, 6]
CHILD_WEIGHTS = [1, 5, 10]
LEARNING_RATES = [0.03, 0.05]
POS_WEIGHTS = [1, 3, 5, 7, 9]
MAX_TREES = 2000
EARLY_STOPPING_ROUNDS = 50
THRESHOLDS = np.round(np.arange(0.05, 0.951, 0.01), 2)

# Use the same tracking database as the existing training notebooks.
TRACKING_URI = "sqlite:///mlflow_churn.db"
EXPERIMENT_NAME = "churn-xgboost-weighted-improvement"
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
SEARCH_ID = uuid4().hex
print("MLflow experiment:", EXPERIMENT_NAME, "Search:", SEARCH_ID)


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: (659028, 44) Validation: (82379, 44)


2026/09/29 15:08:33 INFO mlflow.tracking.fluent: Experiment with name 'churn-xgboost-weighted-improvement' does not exist. Creating a new experiment.


MLflow experiment: churn-xgboost-weighted-improvement Search: 426e1280769a4992993b23d3aaf590bc


## Shared training and evaluation
Feature selection is fitted on training data only. The same transformation is applied to validation data before early stopping. XGBoost's `aucpr` is used for early stopping; sklearn average precision is used for candidate ranking (these are related but not identical metrics). Training metrics help reveal a generalization gap.


In [2]:
feature_cache = {}
records = []

def get_features(k):
    if k not in feature_cache:
        selector = SelectKBest(f_classif, k=k) if k < X_train.shape[1] else None
        train = selector.fit_transform(X_train, y_train) if selector is not None else X_train
        val = selector.transform(X_val) if selector is not None else X_val
        feature_cache[k] = (selector, train, val)
    return feature_cache[k]

def metrics(y, probabilities, threshold=0.5):
    predictions = probabilities >= threshold
    return {
        "accuracy": accuracy_score(y, predictions),
        "balanced_accuracy": balanced_accuracy_score(y, predictions),
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "f1": f1_score(y, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y, probabilities),
        "average_precision": average_precision_score(y, probabilities),
    }

def train_candidate(stage, k, overrides=None, early_stopping=False):
    params = {**BASE_PARAMS, **(overrides or {})}
    if early_stopping:
        params.update(n_estimators=MAX_TREES, early_stopping_rounds=EARLY_STOPPING_ROUNDS)
    selector, train, val = get_features(k)
    model = XGBClassifier(**params)
    fit_options = {"eval_set": [(val, y_val)], "verbose": False} if early_stopping else {}
    run_name = f"{stage}_k{k}_depth{params['max_depth']}_weight{params['scale_pos_weight']}"
    with mlflow.start_run(run_name=run_name, tags={
        "search_id": SEARCH_ID, "stage": stage, "role": "candidate",
        "selection_metric": "val_average_precision",
    }) as run:
        mlflow.log_params({**params, "n_features": k, "decision_threshold": 0.5,
                           "balance": "weighted", "early_stopping_enabled": early_stopping})
        model.fit(train, y_train, **fit_options)
        val_prob = model.predict_proba(val)[:, 1]
        train_prob = model.predict_proba(train)[:, 1]
        pipeline = Pipeline([("feature_selection", selector if selector is not None else "passthrough"),
                             ("model", model)])
        row = {"stage": stage, "n_features": k, **params,
               "effective_trees": model.best_iteration + 1 if early_stopping else params["n_estimators"],
               **{"val_" + key: value for key, value in metrics(y_val, val_prob).items()},
               **{"train_" + key: value for key, value in metrics(y_train, train_prob).items()}}

        row["run_id"] = run.info.run_id
        mlflow.log_param("effective_trees", row["effective_trees"])
        mlflow.log_metrics({key: value for key, value in row.items()
                            if key.startswith(("train_", "val_"))})
        mlflow.sklearn.log_model(pipeline, name="model", serialization_format="cloudpickle")
        if early_stopping:
            for step, score in enumerate(model.evals_result()["validation_0"]["aucpr"]):
                mlflow.log_metric("early_stopping_val_aucpr", score, step=step)
    records.append(row)
    pd.DataFrame(records).to_csv(OUTPUT_DIR / "search_results.csv", index=False)
    print(f"{stage}: k={k}, depth={params['max_depth']}, child={params['min_child_weight']}, "
          f"weight={params['scale_pos_weight']}, AP={row['val_average_precision']:.5f}", flush=True)
    return {"pipeline": pipeline, "probabilities": val_prob, "row": row}

def rank(candidate):
    return candidate["row"]["val_average_precision"]

def show_stage(stage):
    frame = pd.DataFrame(records)
    columns = ["n_features", "max_depth", "min_child_weight", "learning_rate",
               "scale_pos_weight", "effective_trees", "train_average_precision",
               "val_average_precision", "train_f1", "val_f1"]
    display(frame.loc[frame.stage == stage, columns].sort_values("val_average_precision", ascending=False))


## 1. Compare feature counts
Keep baseline hyperparameters fixed. The 25-feature model is retained as the baseline throughout.


In [3]:
feature_candidates = [train_candidate("features", k) for k in FEATURE_COUNTS]
baseline = next(c for c in feature_candidates if c["row"]["n_features"] == 25)
best = max(feature_candidates, key=rank)
chosen_k = best["row"]["n_features"]
show_stage("features")


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/29 15:08:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


features: k=25, depth=3, child=1, weight=7, AP=0.20665


2026/09/29 15:09:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


features: k=44, depth=3, child=1, weight=7, AP=0.20638


,n_features,max_depth,min_child_weight,learning_rate,scale_pos_weight,effective_trees,train_average_precision,val_average_precision,train_f1,val_f1
0,25,3,1,0.03,7,400,0.209325,0.206652,0.269747,0.267731
1,44,3,1,0.03,7,400,0.209520,0.206382,0.269574,0.268506


## 2. Tune depth and minimum child weight
Retain the previous winner if no candidate improves validation average precision.


In [4]:
for depth in DEPTHS:
    for child_weight in CHILD_WEIGHTS:
        candidate = train_candidate("complexity", chosen_k,
                                    {"max_depth": depth, "min_child_weight": child_weight})
        if rank(candidate) > rank(best):
            best = candidate
show_stage("complexity")
complexity_params = {key: best["row"][key] for key in ["max_depth", "min_child_weight"]}


2026/09/29 15:09:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


complexity: k=25, depth=3, child=1, weight=7, AP=0.20665


2026/09/29 15:10:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


complexity: k=25, depth=3, child=5, weight=7, AP=0.20660


2026/09/29 15:10:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

2026/09/29 15:13:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


complexity: k=25, depth=6, child=5, weight=7, AP=0.20377


2026/09/29 15:13:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


complexity: k=25, depth=6, child=10, weight=7, AP=0.20421


,n_features,max_depth,min_child_weight,learning_rate,scale_pos_weight,effective_trees,train_average_precision,val_average_precision,train_f1,val_f1
2,25,3,1,0.03,7,400,0.209325,0.206652,0.269747,0.267731
4,25,3,10,0.03,7,400,0.209269,0.206620,0.269577,0.267691
3,25,3,5,0.03,7,400,0.209294,0.206602,0.269622,0.267359
7,25,4,10,0.03,7,400,0.212412,0.205975,0.271406,0.267172
6,25,4,5,0.03,7,400,0.212442,0.205946,0.271390,0.267150
5,25,4,1,0.03,7,400,0.212590,0.205924,0.271312,0.267029
8,25,5,1,0.03,7,400,0.219068,0.204955,0.274990,0.266388
9,25,5,5,0.03,7,400,0.218731,0.204492,0.275044,0.265145
10,25,5,10,0.03,7,400,0.218539,0.204473,0.274949,0.266478
13,25,6,10,0.03,7,400,0.229344,0.204208,0.281732,0.266345


## 3. Tune learning rate and boosting duration
Early stopping monitors the transformed validation split. Predictions automatically use the best iteration.


In [5]:
for learning_rate in LEARNING_RATES:
    candidate = train_candidate("early_stopping", chosen_k,
                                {**complexity_params, "learning_rate": learning_rate},
                                early_stopping=True)
    if rank(candidate) > rank(best):
        best = candidate
show_stage("early_stopping")
# Freeze boosting duration for the subsequent controlled weight comparison.
weight_params = {key: best["row"][key] for key in ["max_depth", "min_child_weight", "learning_rate"]}
weight_params["n_estimators"] = best["row"]["effective_trees"]


2026/09/29 15:13:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


early_stopping: k=25, depth=3, child=1, weight=7, AP=0.20673


2026/09/29 15:14:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


early_stopping: k=25, depth=3, child=1, weight=7, AP=0.20671


,n_features,max_depth,min_child_weight,learning_rate,scale_pos_weight,effective_trees,train_average_precision,val_average_precision,train_f1,val_f1
14,25,3,1,0.03,7,445,0.209649,0.206725,0.269569,0.268257
15,25,3,1,0.05,7,276,0.209690,0.206713,0.269539,0.267915


## 4. Compare class weights
Train once per weight. Threshold selection in the next step requires no retraining.


In [6]:
for weight in POS_WEIGHTS:
    candidate = train_candidate("weights", chosen_k,
                                {**weight_params, "scale_pos_weight": weight})
    if rank(candidate) > rank(best):
        best = candidate
show_stage("weights")


2026/09/29 15:14:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


weights: k=25, depth=3, child=1, weight=1, AP=0.20670


2026/09/29 15:14:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


weights: k=25, depth=3, child=1, weight=3, AP=0.20663


2026/09/29 15:15:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

2026/09/29 15:15:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


weights: k=25, depth=3, child=1, weight=7, AP=0.20673


2026/09/29 15:15:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


weights: k=25, depth=3, child=1, weight=9, AP=0.20672


,n_features,max_depth,min_child_weight,learning_rate,scale_pos_weight,effective_trees,train_average_precision,val_average_precision,train_f1,val_f1
19,25,3,1,0.03,7,445,0.209649,0.206725,0.269569,0.268257
20,25,3,1,0.03,9,445,0.209541,0.206724,0.257206,0.254972
16,25,3,1,0.03,1,445,0.210192,0.206701,0.007909,0.006812
17,25,3,1,0.03,3,445,0.209756,0.206631,0.151018,0.149169
18,25,3,1,0.03,5,445,0.209664,0.206567,0.259408,0.254327


## 5. Select the threshold and save the candidate
Select the threshold by validation F1, breaking ties by recall and then precision. Better average precision does not guarantee better F1; compare both against the baseline. Model selection uses average precision consistently throughout.


In [7]:
threshold_results = pd.DataFrame([
    {"decision_threshold": float(t), **metrics(y_val, best["probabilities"], t)}
    for t in THRESHOLDS
]).sort_values(["f1", "recall", "precision"], ascending=False)
best_threshold = float(threshold_results.iloc[0]["decision_threshold"])
threshold_results.to_csv(OUTPUT_DIR / "validation_threshold_results.csv", index=False)
comparison = pd.DataFrame({
    "baseline_threshold_0.5": metrics(y_val, baseline["probabilities"], 0.5),
    "candidate_threshold_0.5": metrics(y_val, best["probabilities"], 0.5),
    "candidate_selected_threshold": metrics(y_val, best["probabilities"], best_threshold),
})
comparison.to_csv(OUTPUT_DIR / "validation_comparison.csv")
display(comparison)
display(threshold_results.head(10))
print("Selected threshold:", best_threshold)
print("Selected configuration:", best["row"])

# Persist the already-fitted selector and model; do not refit on validation data.
with (OUTPUT_DIR / "selected_pipeline.pkl").open("wb") as handle:
    pickle.dump(best["pipeline"], handle)
metadata = {
    "selection_metric": "validation_average_precision",
    "threshold_metric": "validation_f1",
    "decision_threshold": best_threshold,
    "configuration_and_metrics": best["row"],
}
(OUTPUT_DIR / "selected_configuration.json").write_text(json.dumps(metadata, indent=2))
print("Saved results and fitted pipeline to", OUTPUT_DIR)

# Separate final-selection run preserves candidate metrics at threshold 0.5.
with mlflow.start_run(run_name="selected_weighted_xgboost", tags={
    "search_id": SEARCH_ID, "role": "selected_model",
    "source_run_id": best["row"]["run_id"],
}) as selected_run:
    selected_run_id = selected_run.info.run_id
    mlflow.log_params({key: value for key, value in best["row"].items()
                       if not key.startswith(("train_", "val_")) and key != "run_id"})
    mlflow.log_params({"decision_threshold": best_threshold,
                       "selection_metric": "val_average_precision", "threshold_metric": "val_f1"})
    mlflow.log_metrics({"val_" + key: value for key, value in
                        metrics(y_val, best["probabilities"], best_threshold).items()})
    mlflow.log_metrics({"baseline_val_" + key: value for key, value in
                        metrics(y_val, baseline["probabilities"], 0.5).items()})
    mlflow.sklearn.log_model(best["pipeline"], name="model", serialization_format="cloudpickle")
    for filename in ["search_results.csv", "validation_threshold_results.csv",
                     "validation_comparison.csv", "selected_configuration.json"]:
        mlflow.log_artifact(str(OUTPUT_DIR / filename))
    mlflow.log_dict({"decision_threshold": best_threshold,
                     "source_run_id": best["row"]["run_id"],
                     "prediction_rule": "predict_proba(X)[:, 1] >= decision_threshold"},
                    "prediction_settings.json")
print("Selected MLflow run:", selected_run_id)
print("Use the saved decision threshold with predict_proba; pipeline.predict uses 0.5.")


,baseline_threshold_0.5,candidate_threshold_0.5,candidate_selected_threshold
accuracy,0.736183,0.735564,0.750604
balanced_accuracy,0.625042,0.625788,0.622423
precision,0.184696,0.184844,0.189535
recall,0.486410,0.488859,0.462537
f1,0.267731,0.268257,0.268887
roc_auc,0.684104,0.684118,0.684118
average_precision,0.206652,0.206725,0.206725


,decision_threshold,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
46,0.51,0.750604,0.622423,0.189535,0.462537,0.268887,0.684118,0.206725
45,0.50,0.735564,0.625788,0.184844,0.488859,0.268257,0.684118,0.206725
48,0.53,0.776666,0.614608,0.198550,0.412463,0.268062,0.684118,0.206725
47,0.52,0.763386,0.618132,0.193316,0.436949,0.268044,0.684118,0.206725
49,0.54,0.788381,0.609942,0.202912,0.387365,0.266319,0.684118,0.206725
44,0.49,0.720208,0.627289,0.179772,0.511386,0.266026,0.684118,0.206725
50,0.55,0.800119,0.605997,0.208679,0.363859,0.265239,0.684118,0.206725
51,0.56,0.811263,0.602540,0.215497,0.342189,0.264453,0.684118,0.206725
43,0.48,0.703881,0.629122,0.175220,0.535872,0.264088,0.684118,0.206725
42,0.47,0.687117,0.630223,0.170811,0.559256,0.261694,0.684118,0.206725


Selected threshold: 0.51
Selected configuration: {'stage': 'early_stopping', 'n_features': 25, 'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 2000, 'subsample': 0.8, 'colsample_bytree': 0.8, 'scale_pos_weight': 7, 'min_child_weight': 1, 'random_state': 42, 'n_jobs': 8, 'eval_metric': 'aucpr', 'early_stopping_rounds': 50, 'effective_trees': 445, 'val_accuracy': 0.7355636752084876, 'val_balanced_accuracy': 0.6257880396053126, 'val_precision': 0.18484399592630313, 'val_recall': 0.48885896180215477, 'val_f1': 0.26825663419549883, 'val_roc_auc': 0.6841178749910369, 'val_average_precision': 0.20672509732768157, 'train_accuracy': 0.7384724169534527, 'train_balanced_accuracy': 0.6264523109136422, 'train_precision': 0.1864036244497975, 'train_recall': 0.48672390309597047, 'train_f1': 0.26956882887922634, 'train_roc_auc': 0.6891313734725989, 'train_average_precision': 0.20964920424036895, 'run_id': 'eea2a1456ecd464999ebc15a50920b8d'}
Saved results and fitted pipeline to data/xgboost_weig

2026/09/29 15:16:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Selected MLflow run: 7a502ac030594b108794e588ab79ce13
Use the saved decision threshold with predict_proba; pipeline.predict uses 0.5.


## 6. Optional final test evaluation
Leave this disabled while experimenting. Once the model and threshold are frozen, set `RUN_FINAL_TEST = True`. This reports baseline and candidate performance without making any selection from test scores. The test set was previously inspected for the baseline; avoid repeated test-driven changes.


In [8]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    with np.load(DATA_PATH) as data:
        X_test, y_test = data["X_test"], data["Y_test"]
    final_rows = []
    for name, candidate, threshold in [
        ("baseline", baseline, 0.5), ("selected_candidate", best, best_threshold)
    ]:
        probabilities = candidate["pipeline"].predict_proba(X_test)[:, 1]
        final_rows.append({"model": name, "decision_threshold": threshold,
                           "n_test": len(y_test), **metrics(y_test, probabilities, threshold)})
        matrix = pd.DataFrame(confusion_matrix(y_test, probabilities >= threshold, labels=[0, 1]),
                              index=["actual_no_churn", "actual_churn"],
                              columns=["predicted_no_churn", "predicted_churn"])
        matrix.to_csv(OUTPUT_DIR / f"{name}_test_confusion_matrix.csv")
        print(name)
        display(matrix)
    test_results = pd.DataFrame(final_rows)
    test_results.to_csv(OUTPUT_DIR / "final_test_results.csv", index=False)
    display(test_results)
    with mlflow.start_run(run_id=selected_run_id):
        for result in final_rows:
            prefix = "baseline_test_" if result["model"] == "baseline" else "test_"
            mlflow.log_metrics({prefix + key: value for key, value in result.items()
                                if key not in ["model", "decision_threshold", "n_test"]})
        mlflow.log_param("n_test", len(y_test))
        for filename in ["final_test_results.csv", "baseline_test_confusion_matrix.csv",
                         "selected_candidate_test_confusion_matrix.csv"]:
            mlflow.log_artifact(str(OUTPUT_DIR / filename), artifact_path="test_evaluation")
else:
    print("Test evaluation disabled. Freeze your candidate before enabling it.")


Test evaluation disabled. Freeze your candidate before enabling it.
